# WWINP Module Testing

This notebook tests the `kika.wwinp` module: reading, writing, querying, and manipulating MCNP weight window files.

Test data: `wwinp_ueki` — a Cartesian mesh WWINP file with 1 particle type and 27 energy bins.

In [ ]:
from pathlib import Path
import numpy as np

from kika.wwinp import read_wwinp, WWINP, write_wwinp
from kika.wwinp.classes.geometry import GeometryAxis, GeometryData
from kika.wwinp.classes.weight_windows import WeightWindowValues
from kika.wwinp._exceptions import WWINPParsingError

DATA_DIR = Path(__file__).parent / "data" if "__file__" in dir() else Path("kika/wwinp/tests/data")
if not DATA_DIR.is_dir():
    # Fallback: resolve relative to this notebook's location
    DATA_DIR = Path("data")
UEKI = DATA_DIR / "wwinp_ueki"
assert UEKI.is_file(), f"Test data not found: {UEKI}"
print(f"Data dir: {DATA_DIR.resolve()}")

## 1. Reading a WWINP file

In [ ]:
ww = read_wwinp(UEKI)
print(ww)

### 1.1 Header verification

In [ ]:
h = ww.header
print(f"if={h.if_}  iv={h.iv}  ni={h.ni}  nr={h.nr}")
print(f"nfx={h.nfx}  nfy={h.nfy}  nfz={h.nfz}  n_spatial={h.n_spatial}")
print(f"origin=({h.x0}, {h.y0}, {h.z0})")
print(f"ncx={h.ncx}  ncy={h.ncy}  ncz={h.ncz}  nwg={h.nwg}")
print(f"mesh_type={h.mesh_type}  time_dep={h.has_time_dependency}")
print(f"ne={h.ne}  nt={h.nt}")

# File declares ni=2 but second particle has ne=0 → removed by verify_and_correct
assert h.if_ == 1
assert h.iv == 1  # no time dependency
assert h.ni == 1  # second particle pruned (ne=0)
assert h.nr == 10 # rectangular
assert h.nwg == 1 # cartesian
assert h.nfx == 8 and h.nfy == 7 and h.nfz == 7
assert h.n_spatial == 8 * 7 * 7
print("\n✓ Header OK")

### 1.2 Geometry verification

In [ ]:
fm = ww.geometry.fine_mesh
labels = ww.geometry.axis_labels
print(f"Axis labels: {labels}")

for label in labels:
    grid = fm[label]
    print(f"  {label}: [{grid[0]:.2f}, {grid[-1]:.2f}]  bins={len(grid)-1}  points={len(grid)}")

# Fine mesh should have nfx+1, nfy+1, nfz+1 boundary points
assert len(fm["x"]) == h.nfx + 1
assert len(fm["y"]) == h.nfy + 1
assert len(fm["z"]) == h.nfz + 1

# Check origin
assert fm["x"][0] == -25.0
assert fm["y"][0] == -40.0
assert fm["z"][0] == -40.0

# Coarse mesh
cm = ww.geometry.coarse_mesh
for label in labels:
    print(f"  coarse {label}: {cm[label]}")

print("\n✓ Geometry OK")

### 1.3 Energy bins and value arrays

In [ ]:
print(f"Energy bins for particle 0: {len(ww.energy_bins[0])} bins")
print(f"  range: [{ww.energy_bins[0][0]:.3e}, {ww.energy_bins[0][-1]:.3e}]")
print(f"  values: {ww.energy_bins[0]}")

assert 0 in ww.energy_bins
assert len(ww.energy_bins[0]) == 27  # ne[0] = 27

# 5-D array: (nt, ne, nfz, nfy, nfx)
arr = ww.values.ww_values[0]
print(f"\nValue array shape: {arr.shape}")
assert arr.ndim == 5
assert arr.shape == (1, 27, 7, 7, 8)
assert np.all(arr >= 0), "All values should be non-negative"

pos = arr[arr > 0]
print(f"  min={pos.min():.3e}  max={pos.max():.3e}  nonzero={pos.size}/{arr.size} ({100*pos.size/arr.size:.1f}%)")
print("\n✓ Energy bins & values OK")

### 1.4 Error handling

In [ ]:
import tempfile, os

# Missing file
try:
    read_wwinp("/tmp/nonexistent_wwinp_file")
    assert False, "Should have raised FileNotFoundError"
except FileNotFoundError:
    print("✓ FileNotFoundError for missing file")

# Empty file
with tempfile.NamedTemporaryFile(mode="w", suffix=".wwinp", delete=False) as f:
    f.write("")
    empty_path = f.name
try:
    read_wwinp(empty_path)
    assert False, "Should have raised WWINPParsingError"
except WWINPParsingError:
    print("✓ WWINPParsingError for empty file")
finally:
    os.unlink(empty_path)

## 2. Geometry classes

In [ ]:
# Single segment: [0, 10] with 5 fine bins → [0, 2, 4, 6, 8, 10]
axis = GeometryAxis(
    origin=0.0,
    q=np.array([1.0]),
    p=np.array([10.0]),
    s=np.array([5], dtype=np.int32),
)
np.testing.assert_allclose(axis.fine_mesh, [0, 2, 4, 6, 8, 10])
np.testing.assert_allclose(axis.coarse_mesh, [0, 10])
assert axis.n_fine == 5
print(f"Single segment: fine_mesh={axis.fine_mesh}, coarse_mesh={axis.coarse_mesh}")

# Two segments: [-10, 0] with 2 bins + [0, 10] with 5 bins
axis2 = GeometryAxis(
    origin=-10.0,
    q=np.array([1.0, 1.0]),
    p=np.array([0.0, 10.0]),
    s=np.array([2, 5], dtype=np.int32),
)
assert axis2.fine_mesh[0] == -10.0
assert axis2.fine_mesh[-1] == 10.0
assert len(axis2.fine_mesh) == 2 + 5 + 1  # 8 boundaries for 7 bins
assert axis2.n_fine == 7
print(f"Two segments: fine_mesh={axis2.fine_mesh}")

print("\n✓ GeometryAxis OK")

In [ ]:
# Axis labels per geometry type (MCNP Appendix A convention)
def make_axis(o, e, n):
    return GeometryAxis(origin=o, q=np.array([1.0]), p=np.array([e]), s=np.array([n], dtype=np.int32))

gd_cart = GeometryData(axes=(make_axis(0,1,2), make_axis(0,1,2), make_axis(0,1,2)), geometry_type=1)
gd_cyl  = GeometryData(axes=(make_axis(0,1,2), make_axis(0,1,2), make_axis(0,1,2)), geometry_type=2)
gd_sph  = GeometryData(axes=(make_axis(0,1,2), make_axis(0,1,2), make_axis(0,1,2)), geometry_type=3)

assert gd_cart.axis_labels == ("x", "y", "z")
assert gd_cyl.axis_labels == ("r", "z", "theta")
assert gd_sph.axis_labels == ("r", "phi", "theta")  # MCNP: (r, φ, θ)
print(f"Cartesian:   {gd_cart.axis_labels}")
print(f"Cylindrical: {gd_cyl.axis_labels}")
print(f"Spherical:   {gd_sph.axis_labels}")

# Fine mesh dict
gd = GeometryData(
    axes=(make_axis(0, 10, 5), make_axis(-5, 5, 4), make_axis(0, 20, 10)),
    geometry_type=1,
)
assert len(gd.fine_mesh["x"]) == 6
assert len(gd.fine_mesh["y"]) == 5
assert len(gd.fine_mesh["z"]) == 11
print(f"\nfine_mesh sizes: x={len(gd.fine_mesh['x'])}, y={len(gd.fine_mesh['y'])}, z={len(gd.fine_mesh['z'])}")

print("\n✓ GeometryData OK")

## 3. Weight window operations

In [ ]:
# --- Multiply ---
vals = WeightWindowValues(ww_values={0: np.full((1, 2, 3, 3, 3), 2.0)})
vals.multiply(3.0)
assert np.all(vals.ww_values[0] == 6.0)
print("multiply(3.0) on uniform 2.0 → 6.0 ✓")

# Multi-particle: only scale particle 0
vals2 = WeightWindowValues(ww_values={
    0: np.ones((1, 1, 2, 2, 2)),
    1: np.ones((1, 1, 2, 2, 2)) * 5.0,
})
vals2.multiply(2.0, particles=0)
assert np.allclose(vals2.ww_values[0], 2.0)
assert np.allclose(vals2.ww_values[1], 5.0)
print("multiply(2.0, particles=0): particle 0 scaled, particle 1 untouched ✓")

# Invalid particle
try:
    vals.multiply(2.0, particles=99)
    assert False
except ValueError as e:
    print(f"Invalid particle → ValueError: {e} ✓")

In [ ]:
# --- Soften ---
vals_s = WeightWindowValues(ww_values={0: np.full((1, 1, 3, 3, 3), 4.0)})
vals_s.soften(0.5)
np.testing.assert_allclose(vals_s.ww_values[0], 2.0)
print("soften(0.5) on uniform 4.0 → 2.0 ✓")

# --- Ratio threshold ---
# Uniform array → no changes
vals_u = WeightWindowValues(ww_values={0: np.ones((1, 1, 3, 3, 3))})
changes = vals_u.apply_ratio_threshold(5.0)
assert changes == 0
print(f"Uniform array: {changes} changes ✓")

# Spike in center → neighbours get zeroed (not the spike itself)
arr = np.ones((1, 1, 3, 3, 3))
arr[0, 0, 1, 1, 1] = 100.0
vals_r = WeightWindowValues(ww_values={0: arr})
changes = vals_r.apply_ratio_threshold(5.0)
assert changes > 0
# The spike itself has ratio = max_neighbor/center = 1/100 = 0.01 → below threshold
assert vals_r.ww_values[0][0, 0, 1, 1, 1] == 100.0, "Spike should be untouched"
# Its neighbours have ratio = 100/1 = 100 → above threshold → zeroed
assert vals_r.ww_values[0][0, 0, 1, 1, 0] == 0.0, "Neighbour should be zeroed"
print(f"Spike array: {changes} cells zeroed, spike untouched ✓")

## 4. Round-trip: read → write → read

In [ ]:
import tempfile

ww1 = read_wwinp(UEKI)

with tempfile.NamedTemporaryFile(suffix=".wwinp", delete=False) as f:
    out_path = f.name

ww1.write(out_path, overwrite=True)
ww2 = read_wwinp(out_path)

# Header
assert ww2.header.if_ == ww1.header.if_
assert ww2.header.iv == ww1.header.iv
assert ww2.header.ni == ww1.header.ni
assert ww2.header.nr == ww1.header.nr
assert ww2.header.nfx == ww1.header.nfx
assert ww2.header.nfy == ww1.header.nfy
assert ww2.header.nfz == ww1.header.nfz
assert ww2.header.nwg == ww1.header.nwg
assert ww2.header.ne == ww1.header.ne
assert ww2.header.nt == ww1.header.nt
print("Header matches ✓")

# Geometry
for i in range(3):
    ax1 = ww1.geometry.axes[i]
    ax2 = ww2.geometry.axes[i]
    np.testing.assert_allclose(ax2.origin, ax1.origin, rtol=1e-4)
    np.testing.assert_allclose(ax2.p, ax1.p, rtol=1e-4)
    np.testing.assert_array_equal(ax2.s, ax1.s)
print("Geometry matches ✓")

# Energy bins
for p in range(ww1.header.ni):
    np.testing.assert_allclose(ww2.energy_bins[p], ww1.energy_bins[p], rtol=1e-4)
print("Energy bins match ✓")

# Weight window values
for p in range(ww1.header.ni):
    np.testing.assert_allclose(
        ww2.values.ww_values[p], ww1.values.ww_values[p], rtol=1e-4, atol=1e-10
    )
    max_rel_err = np.max(np.abs(ww2.values.ww_values[p] - ww1.values.ww_values[p]) / 
                         np.maximum(np.abs(ww1.values.ww_values[p]), 1e-30))
    print(f"  Particle {p}: max relative error = {max_rel_err:.2e}")
print("Weight window values match ✓")

os.unlink(out_path)
print("\n✓ Round-trip OK")

## 5. Query interface

In [ ]:
ww = read_wwinp(UEKI)

# Query a spatial subregion
result = ww.query(particle=0, x=(-10, 10), y=(-5, 5), z=(-5, 5))
print(f"Query result: {len(result.particle_types)} particle(s)")
print(f"  ww shape: {result.ww_values[0].shape}")
print(f"  spatial intervals: {list(result.spatial_intervals.keys())}")

for label, (starts, ends) in result.spatial_intervals.items():
    print(f"    {label}: {len(starts)} bins, range [{starts[0]:.1f}, {ends[-1]:.1f}]")

# Full query (no filters) should return all data
result_all = ww.query(particle=0)
assert result_all.ww_values[0].shape == ww.values.ww_values[0].shape
print(f"\nFull query shape: {result_all.ww_values[0].shape} == original shape ✓")

print("\n✓ Query OK")

## 6. Copy and delegated operations on WWINP

In [ ]:
ww = read_wwinp(UEKI)
original_max = ww.values.ww_values[0].max()

# Deep copy — mutations on copy should not affect original
ww_copy = ww.copy()
ww_copy.multiply(2.0)
assert ww.values.ww_values[0].max() == original_max, "Original must be unchanged"
assert ww_copy.values.ww_values[0].max() == original_max * 2.0
print(f"copy + multiply: original max={original_max:.3e}, copy max={ww_copy.values.ww_values[0].max():.3e} ✓")

# Soften via WWINP delegation
ww_soft = ww.copy()
ww_soft.soften(0.5)
# x^0.5 < x for x > 1, so max should decrease
assert ww_soft.values.ww_values[0].max() < original_max
print(f"soften(0.5): max {original_max:.3e} → {ww_soft.values.ww_values[0].max():.3e} ✓")

# __str__ and __repr__
s = str(ww)
assert "cartesian" in s
assert "Particle 0" in s
r = repr(ww)
assert "WWINP(" in r
print(f"\nrepr: {r}")
print("\n✓ WWINP operations OK")

## 7. Ratio statistics (pure numpy)

In [ ]:
from kika.wwinp.utils import calculate_max_ratio_array, calculate_ratios_stats

# Uniform array → all ratios = 1
uniform = np.ones((5, 5, 5))
ratios = calculate_max_ratio_array(uniform)
assert np.all(ratios == 1.0)
avg, mx = calculate_ratios_stats(uniform)
assert avg == 1.0 and mx == 1.0
print(f"Uniform: avg_ratio={avg:.2f}, max_ratio={mx:.2f} ✓")

# Gradient along x: values 1,2,3,4,5
gradient = np.ones((1, 1, 5))
gradient[0, 0, :] = [1, 2, 3, 4, 5]
avg_g, max_g = calculate_ratios_stats(gradient)
# Adjacent ratios: 2/1=2, 3/2=1.5, 4/3=1.33, 5/4=1.25
expected_avg = np.mean([2, 1.5, 4/3, 5/4])
assert np.isclose(avg_g, expected_avg, rtol=1e-10)
assert np.isclose(max_g, 2.0)
print(f"Gradient [1..5]: avg_ratio={avg_g:.4f}, max_ratio={max_g:.2f} ✓")

# Real data from Ueki
ww = read_wwinp(UEKI)
spatial = ww.values.ww_values[0][0, 0]  # first time, first energy → (nfz, nfy, nfx)
avg_r, max_r = calculate_ratios_stats(spatial)
print(f"\nUeki (energy bin 0): avg_ratio={avg_r:.2f}, max_ratio={max_r:.2f}")
print("\n✓ Ratio stats OK")